# Neural Additive Model (NAM) for Seismic Horizontal Response Prediction

This notebook implements a **Neural Additive Model (NAM)** to predict `Y_horizontal` (geometric mean horizontal ground motion) as a time-series output across multiple spectral periods.

### Architecture Overview
- Each input feature $x_i$ is passed through its own **sub-network** $f_i(x_i)$
- Final prediction: $\hat{y} = \beta_0 + \sum_{i=1}^{p} f_i(x_i)$
- Outputs are predicted simultaneously for **all periods** (multi-output regression)
- Fully interpretable: feature contributions can be visualised per period

---

## 1. Install & Import Dependencies

In [ ]:
# Uncomment if not already installed
# !pip install torch scipy scikit-learn matplotlib seaborn tqdm

In [ ]:
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Load & Inspect Data

In [ ]:
mat_path = r"A:\Github\Seismic_data_analysis\Seismic-Data-analysis\Data\Inelastic_esm_inputs_final.mat"
print(f'Loading: {mat_path} ...')
mat = scipy.io.loadmat(mat_path)
print('Keys:', [k for k in mat.keys() if not k.startswith('__')])

In [ ]:
feature_cols = [
    'IP1_Event_Magnitude',
    'IP2_R_epi',
    'IP3_Vs30',
    'IP4_Event_Depth',
    'IP5_Fault',
    'IP6_Mu',
    'IP7_ksi',
    'IP8_p',
]
feature_names = ['Magnitude', 'R_epi', 'Vs30', 'Depth', 'Fault', 'Mu', 'Ksi', 'p']

X      = np.column_stack([mat[c].ravel().astype(np.float32) for c in feature_cols])
Y_hor  = mat['U_geomean_Hor'].astype(np.float32)
Y_vert = mat['Vertical'].astype(np.float32)
period = mat['period'].ravel()

n_samples, n_features = X.shape
n_periods = Y_hor.shape[1] if Y_hor.ndim > 1 else 1

print(f'X shape        : {X.shape}')
print(f'Y Horizontal   : {Y_hor.shape}')
print(f'Y Vertical     : {Y_vert.shape}')
print(f'Periods        : {period}')
print(f'n_samples      : {n_samples}  |  n_features: {n_features}  |  n_periods: {n_periods}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.ravel()
for i, (name, ax) in enumerate(zip(feature_names, axes)):
    ax.hist(X[:, i], bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Log-transform check: Y_hor is typically log-normal in seismology
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(Y_hor[:, 0], bins=50, color='salmon', edgecolor='white')
axes[0].set_title('Y_hor (Period 0) — Raw', fontweight='bold')
axes[1].hist(np.log(Y_hor[:, 0] + 1e-8), bins=50, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Y_hor (Period 0) — Log-transformed', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Response spectra — mean & std over samples
fig, ax = plt.subplots(figsize=(10, 5))
mean_spec = np.mean(Y_hor, axis=0)
std_spec  = np.std(Y_hor, axis=0)
ax.plot(period, mean_spec, color='royalblue', lw=2, label='Mean')
ax.fill_between(period, mean_spec - std_spec, mean_spec + std_spec,
                alpha=0.3, color='royalblue', label='±1 Std')
ax.set_xlabel('Period (s)', fontsize=12)
ax.set_ylabel('Y_horizontal', fontsize=12)
ax.set_title('Horizontal Response Spectrum (All Samples)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Log-transform target (standard in GMPE/seismic ML literature)
Y_log = np.log(Y_hor + 1e-8).astype(np.float32)

# Standardise features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X).astype(np.float32)

# Standardise log-target
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y_log).astype(np.float32)

print('X_scaled shape:', X_scaled.shape)
print('Y_scaled shape:', Y_scaled.shape)

In [ ]:
# Train / Val / Test split  (70 / 15 / 15)
dataset = TensorDataset(
    torch.tensor(X_scaled),
    torch.tensor(Y_scaled)
)

n_total = len(dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH = 256
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

print(f'Train: {n_train}  |  Val: {n_val}  |  Test: {n_test}')

## 5. Neural Additive Model Architecture

Each feature $x_i$ passes through an independent sub-network $f_i$ parameterised by an MLP with **ExU** (Exponential Linear Unit) activations (as per the original NAM paper by Agarwal et al., 2021).

$$\hat{y}(T) = \beta_0(T) + \sum_{i=1}^{8} f_i(x_i; \theta_i)$$

where $T$ denotes spectral period.

In [ ]:
class ExU(nn.Module):
    """Exponential Unit activation as in NAM paper (Agarwal et al., 2021)."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return (x - self.bias) * torch.exp(self.weight)


class FeatureNet(nn.Module):
    """Per-feature sub-network: maps scalar x_i -> shape function vector of length n_out."""
    def __init__(self, hidden_units, n_out, dropout=0.1):
        super().__init__()
        layers = []
        in_dim = 1
        for i, h in enumerate(hidden_units):
            if i == 0:
                layers.append(ExU(in_dim, h))
            else:
                layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (batch,) -> unsqueeze to (batch, 1)
        return self.net(x.unsqueeze(-1))   # (batch, n_out)


class NAM(nn.Module):
    """
    Neural Additive Model.
    n_features  : number of input features
    n_periods   : number of target spectral periods (multi-output)
    hidden_units: list of hidden layer widths for each FeatureNet
    dropout     : dropout probability
    """
    def __init__(self, n_features, n_periods, hidden_units=(128, 128, 64), dropout=0.1):
        super().__init__()
        self.feature_nets = nn.ModuleList([
            FeatureNet(hidden_units, n_periods, dropout) for _ in range(n_features)
        ])
        self.bias = nn.Parameter(torch.zeros(n_periods))   # intercept per period

    def forward(self, x):
        # x: (batch, n_features)
        contributions = [net(x[:, i]) for i, net in enumerate(self.feature_nets)]
        # Each contribution: (batch, n_periods)
        out = torch.stack(contributions, dim=0).sum(dim=0) + self.bias
        return out   # (batch, n_periods)

    def get_contributions(self, x):
        """Return per-feature contributions. Shape: (n_features, batch, n_periods)"""
        with torch.no_grad():
            return torch.stack([net(x[:, i]) for i, net in enumerate(self.feature_nets)], dim=0)


# Instantiate
model = NAM(
    n_features=n_features,
    n_periods=n_periods,
    hidden_units=(128, 128, 64),
    dropout=0.15
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## 6. Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        total_loss += criterion(pred, yb).item() * len(xb)
    return total_loss / len(loader.dataset)

In [ ]:
# ----- Hyperparameters -----
LR          = 1e-3
EPOCHS      = 200
PATIENCE    = 20   # early stopping patience
# ---------------------------

criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5, verbose=True)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_ctr  = 0
best_state    = None

pbar = tqdm(range(1, EPOCHS + 1), desc='Training')
for epoch in pbar:
    tl = train_epoch(model, train_loader, optimizer, criterion)
    vl = eval_epoch(model, val_loader, criterion)
    scheduler.step(vl)
    train_losses.append(tl)
    val_losses.append(vl)
    pbar.set_postfix({'train_loss': f'{tl:.5f}', 'val_loss': f'{vl:.5f}'})

    # Early stopping
    if vl < best_val_loss:
        best_val_loss = vl
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nBest validation loss: {best_val_loss:.5f}')

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train Loss', color='royalblue')
ax.plot(val_losses,   label='Val Loss',   color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training & Validation Loss', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluation on Test Set

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, targets = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        targets.append(yb.numpy())
    return np.vstack(preds), np.vstack(targets)


Y_pred_scaled, Y_true_scaled = predict_all(model, test_loader)

# Inverse transform: scaled -> log -> original space
Y_pred_log = scaler_Y.inverse_transform(Y_pred_scaled)
Y_true_log = scaler_Y.inverse_transform(Y_true_scaled)
Y_pred_orig = np.exp(Y_pred_log)
Y_true_orig = np.exp(Y_true_log)

# Metrics per period
rmse_per_period = np.sqrt(mean_squared_error(Y_true_log, Y_pred_log, multioutput='raw_values'))
r2_per_period   = r2_score(Y_true_log, Y_pred_log, multioutput='raw_values')

print(f'Mean RMSE (log space): {rmse_per_period.mean():.4f}')
print(f'Mean R²   (log space): {r2_per_period.mean():.4f}')

In [ ]:
# R² and RMSE across periods
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(period, r2_per_period, 'o-', color='royalblue', lw=2)
ax1.axhline(1.0, color='gray', ls='--', lw=1)
ax1.set_xlabel('Period (s)')
ax1.set_ylabel('R²')
ax1.set_title('R² per Spectral Period', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(period, rmse_per_period, 's-', color='tomato', lw=2)
ax2.set_xlabel('Period (s)')
ax2.set_ylabel('RMSE (log scale)')
ax2.set_title('RMSE per Spectral Period', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual — select a few periods to visualise
plot_period_indices = [0, len(period)//4, len(period)//2, -1]
fig, axes = plt.subplots(1, len(plot_period_indices), figsize=(16, 4))

for ax, pi in zip(axes, plot_period_indices):
    ax.scatter(Y_true_log[:, pi], Y_pred_log[:, pi], alpha=0.3, s=10, color='steelblue')
    lim_min = min(Y_true_log[:, pi].min(), Y_pred_log[:, pi].min())
    lim_max = max(Y_true_log[:, pi].max(), Y_pred_log[:, pi].max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5, label='1:1 line')
    ax.set_xlabel('Actual (log)')
    ax.set_ylabel('Predicted (log)')
    ax.set_title(f'T = {period[pi]:.2f}s  |  R²={r2_per_period[pi]:.3f}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual (log space) — Selected Periods', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 8. Interpretability — NAM Shape Functions

A key advantage of NAMs: each feature's contribution can be plotted independently for any target period.

In [ ]:
def plot_shape_functions(model, scaler_X, feature_names, period, period_indices=None, n_grid=300):
    """
    Plot NAM shape functions f_i(x_i) for selected spectral periods.
    period_indices : list of period column indices to overlay (None = first period only)
    """
    if period_indices is None:
        period_indices = [0]

    model.eval()
    n_feat = len(feature_names)
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(period_indices)))

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.ravel()

    for fi in range(n_feat):
        ax = axes[fi]
        # Grid in standardised space
        x_grid_std = np.linspace(-3, 3, n_grid).astype(np.float32)
        x_tensor   = torch.tensor(x_grid_std).to(DEVICE)

        with torch.no_grad():
            shape_vals = model.feature_nets[fi](x_tensor).cpu().numpy()  # (n_grid, n_periods)

        # Back-transform x to original scale for the x-axis
        x_orig = x_grid_std * scaler_X.scale_[fi] + scaler_X.mean_[fi]

        for ci, pi in enumerate(period_indices):
            ax.plot(x_orig, shape_vals[:, pi], color=colors[ci],
                    lw=1.8, label=f'T={period[pi]:.2f}s')

        ax.axhline(0, color='gray', ls='--', lw=0.8)
        ax.set_title(feature_names[fi], fontweight='bold')
        ax.set_xlabel('Feature value')
        ax.set_ylabel('f(x)')
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7)

    plt.suptitle('NAM Shape Functions per Feature  (log-scaled target space)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Select 4 representative periods to overlay
sel_periods = [
    0,
    len(period) // 4,
    len(period) // 2,
    len(period) - 1
]
plot_shape_functions(model, scaler_X, feature_names, period, period_indices=sel_periods)

## 9. Feature Importance (Variance of Shape Functions)

In [ ]:
# Compute feature importance as variance of shape function evaluated on test data
# Build test X tensor
X_test_np  = np.vstack([xb.numpy() for xb, _ in test_loader])
X_test_t   = torch.tensor(X_test_np).to(DEVICE)

contribs = model.get_contributions(X_test_t).cpu().numpy()  # (n_features, n_test, n_periods)

# Variance over test samples, averaged over periods
importance = contribs.var(axis=1).mean(axis=1)  # (n_features,)
importance_norm = importance / importance.sum()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feature_names, importance_norm * 100, color=plt.cm.RdYlGn(importance_norm))
ax.set_xlabel('Relative Importance (%)')
ax.set_title('NAM Feature Importance\n(Variance of Shape Functions, Averaged Over Periods)',
             fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, importance_norm * 100):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 10. Predicted Response Spectrum vs Actual

In [ ]:
# Pick a few random test samples and compare predicted vs actual spectra
np.random.seed(0)
sample_ids = np.random.choice(len(Y_true_orig), size=5, replace=False)

fig, ax = plt.subplots(figsize=(11, 5))
palette = plt.cm.tab10.colors

for k, sid in enumerate(sample_ids):
    ax.plot(period, Y_true_orig[sid], color=palette[k], lw=2, label=f'Actual #{sid}')
    ax.plot(period, Y_pred_orig[sid], color=palette[k], lw=1.5,
            ls='--', marker='o', markersize=3, label=f'Predicted #{sid}')

ax.set_xlabel('Period (s)', fontsize=12)
ax.set_ylabel('Y_horizontal', fontsize=12)
ax.set_title('Predicted vs Actual Response Spectra (Test Samples)', fontweight='bold', fontsize=13)
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Save Model

In [ ]:
import pickle, os

save_dir = r"A:\Github\Seismic_data_analysis\Seismic-Data-analysis\Models"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_dir, 'NAM_Hor.pt'))

with open(os.path.join(save_dir, 'scalers_NAM_Hor.pkl'), 'wb') as f:
    pickle.dump({'scaler_X': scaler_X, 'scaler_Y': scaler_Y}, f)

print('Model and scalers saved successfully!')

## 12. Load & Inference (Template)

In [ ]:
# ----- Load saved model for inference -----
import pickle

with open(os.path.join(save_dir, 'scalers_NAM_Hor.pkl'), 'rb') as f:
    scalers = pickle.load(f)

model_loaded = NAM(n_features=n_features, n_periods=n_periods,
                   hidden_units=(128, 128, 64), dropout=0.15).to(DEVICE)
model_loaded.load_state_dict(torch.load(os.path.join(save_dir, 'NAM_Hor.pt'), map_location=DEVICE))
model_loaded.eval()

# Example: single sample prediction
# new_sample = np.array([[Mw, R_epi, Vs30, depth, fault, mu, ksi, p]], dtype=np.float32)
# new_scaled  = scalers['scaler_X'].transform(new_sample)
# with torch.no_grad():
#     pred_scaled = model_loaded(torch.tensor(new_scaled).to(DEVICE)).cpu().numpy()
# pred_log    = scalers['scaler_Y'].inverse_transform(pred_scaled)
# pred_orig   = np.exp(pred_log)
# print('Predicted spectrum:', pred_orig)

print('Model loaded successfully. Ready for inference.')

---
## Summary

| Step | Details |
|------|----------|
| **Model** | Neural Additive Model (NAM) with ExU activations |
| **Architecture** | 8 independent FeatureNets → additive combination |
| **Target** | `log(Y_horizontal)` across all spectral periods (multi-output) |
| **Interpretability** | Per-feature shape functions visualised for any period |
| **Training** | Adam + ReduceLROnPlateau + Early Stopping |
| **Metrics** | RMSE & R² per spectral period |

> **Reference**: Agarwal, R., Frosst, N., Zhang, X., Caruana, R., & Hinton, G. E. (2021). *Neural Additive Models: Interpretable Machine Learning with Neural Nets.* NeurIPS 2021.